In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from timm.models.layers import to_2tuple, trunc_normal_
from collections import OrderedDict
from torch.utils.data import DataLoader
from torchvision import transforms
import shutil
from datasets import load_dataset
from PIL import Image

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:
def rounding_shift_right(tensor, shift_amt):
    sign = tensor.sign()
    tensor = tensor.abs()
    mask = (2 ** shift_amt.abs().to(torch.int32)) - 1
    lsb = tensor & mask
    tensor = tensor >> shift_amt.abs()

    if log_file is not None:
        print(f"--- tensor >> shift_amt.abs ---", file=log_file)
        print(f"tensor shape: {tensor.shape}", file=log_file)
        print(f"tensor max: {tensor.max().item()}", file=log_file)
        print(f"tensor min: {tensor.min().item()}", file=log_file)
        print(f"tensor: {tensor}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    threshold = (mask >> 1) + 1
    is_greater_than_half = lsb > threshold
    is_half_and_odd = (lsb == threshold) & ((tensor & 1) == 1)
    should_round_up = is_greater_than_half | is_half_and_odd
    tensor = torch.where(
            should_round_up,
            tensor + 1,
            tensor
        )
    tensor = tensor * sign

    if log_file is not None:
        print(f"--- tensor round by bits ---", file=log_file)
        print(f"tensor shape: {tensor.shape}", file=log_file)
        print(f"tensor max: {tensor.max().item()}", file=log_file)
        print(f"tensor min: {tensor.min().item()}", file=log_file)
        print(f"tensor: {tensor}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    return tensor

def shift_left(tensor, shift_amt):
    tensor = tensor << shift_amt

    if log_file is not None:
        print(f"--- tensor << shift_amt ---", file=log_file)
        print(f"tensor shape: {tensor.shape}", file=log_file)
        print(f"tensor max: {tensor.max().item()}", file=log_file)
        print(f"tensor min: {tensor.min().item()}", file=log_file)
        print(f"tensor: {tensor}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    return tensor


def quantize(tensor: torch.Tensor, scale: torch.Tensor, is_input=True) -> torch.Tensor:

    Q_MIN = -127
    Q_MAX = 127
    scale_view = scale.squeeze()

    if log_file is not None:
        print(f"--- scale ---", file=log_file)
        print(f"scale shape: {scale_view.shape}", file=log_file)
        print(f"scale: {scale_view}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    if is_input:
        quantized_tensor = tensor / scale_view

        if log_file is not None:
            print(f"--- tensor / scale ---", file=log_file)
            print(f"tensor shape: {quantized_tensor.shape}", file=log_file)
            print(f"tensor max: {quantized_tensor.max().item()}", file=log_file)
            print(f"tensor min: {quantized_tensor.min().item()}", file=log_file)
            print(f"tensor: {quantized_tensor}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        quantized_tensor = torch.round(quantized_tensor)

        if log_file is not None:
            print(f"--- tensor round ---", file=log_file)
            print(f"tensor shape: {quantized_tensor.shape}", file=log_file)
            print(f"tensor max: {quantized_tensor.max().item()}", file=log_file)
            print(f"tensor min: {quantized_tensor.min().item()}", file=log_file)
            print(f"tensor: {quantized_tensor}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        quantized_tensor = torch.clamp(quantized_tensor, Q_MIN, Q_MAX)

        if log_file is not None:
            print(f"--- tensor clamp ---", file=log_file)
            print(f"tensor shape: {quantized_tensor.shape}", file=log_file)
            print(f"tensor max: {quantized_tensor.max().item()}", file=log_file)
            print(f"tensor min: {quantized_tensor.min().item()}", file=log_file)
            print(f"tensor: {quantized_tensor}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        return quantized_tensor.to(torch.int8)
    else:
        # shift_view = torch.log2(scale_view).to(torch.int8)

        tensor = torch.round(tensor).to(torch.int32)
        quantized_tensor = torch.where(
            scale_view > 0,
            rounding_shift_right(tensor, scale_view.abs()),
            shift_left(tensor, scale_view.abs())
        )
        quantized_tensor = torch.round(quantized_tensor)
        quantized_tensor = torch.clamp(quantized_tensor, Q_MIN, Q_MAX)

        if log_file is not None:
            print(f"--- quantized_tensor clamp ---", file=log_file)
            print(f"quantized_tensor shape: {quantized_tensor.shape}", file=log_file)
            print(f"quantized_tensor max: {quantized_tensor.max().item()}", file=log_file)
            print(f"quantized_tensor min: {quantized_tensor.min().item()}", file=log_file)
            print(f"quantized_tensor: {quantized_tensor}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        return quantized_tensor.to(torch.int8)


def dequantize(quantized_tensor: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:

    Q_MIN = -127
    Q_MAX = 127
    scale_view = scale.squeeze()

    if log_file is not None:
        print(f"--- scale ---", file=log_file)
        print(f"scale shape: {scale_view.shape}", file=log_file)
        print(f"scale: {scale_view}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    dequantized_tensor = quantized_tensor * scale_view
    return dequantized_tensor

def log2_quantize(tensor):

    log2_tensor = torch.round(torch.log2(tensor)).abs()

    if log_file is not None:
        print(f"--- log2_tensor.abs ---", file=log_file)
        print(f"log2_tensor shape: {log2_tensor.shape}", file=log_file)
        print(f"log2_tensor max: {log2_tensor.max().item()}", file=log_file)
        print(f"log2_tensor min: {log2_tensor.min().item()}", file=log_file)
        print(f"log2_tensor: {log2_tensor}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    log2_tensor = torch.clamp(log2_tensor, 0, 127).to(torch.int8)

    if log_file is not None:
        print(f"--- log2_tensor clamp ---", file=log_file)
        print(f"log2_tensor shape: {log2_tensor.shape}", file=log_file)
        print(f"log2_tensor max: {log2_tensor.max().item()}", file=log_file)
        print(f"log2_tensor min: {log2_tensor.min().item()}", file=log_file)
        print(f"log2_tensor: {log2_tensor}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    power = 7 - log2_tensor

    if log_file is not None:
        print(f"--- 7 - log2_tensor ---", file=log_file)
        print(f"power shape: {power.shape}", file=log_file)
        print(f"power max: {power.max()}", file=log_file)
        print(f"power min: {power.min()}", file=log_file)
        print(f"power: {power}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    output = 1 << power.to(torch.int32)

    if log_file is not None:
        print(f"--- 1 << power ---", file=log_file)
        print(f"output shape: {output.shape}", file=log_file)
        print(f"output max: {output.max()}", file=log_file)
        print(f"output min: {output.min()}", file=log_file)
        print(f"output: {output}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    output = torch.round(output)
    output = torch.clamp(output, -127, 127)

    if log_file is not None:
        print(f"--- output clamp ---", file=log_file)
        print(f"output shape: {output.shape}", file=log_file)
        print(f"output max: {output.max()}", file=log_file)
        print(f"output min: {output.min()}", file=log_file)
        print(f"output: {output}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    return output.to(torch.int8)

In [16]:
def softmax(x, dim=-1):

    x = x.float()

    max_x = torch.max(x, dim=dim, keepdim=True)[0]

    if log_file is not None:
        print(f"--- softmax: torch.max(x) ---", file=log_file)
        print(f"max_x shape: {max_x.shape}", file=log_file)
        print(f"max_x max: {max_x.max()}", file=log_file)
        print(f"max_x min: {max_x.min()}", file=log_file)
        print(f"max_x: {max_x}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    x = x - max_x
    x = torch.clamp(x, -127, 127).to(torch.int8)

    if log_file is not None:
        print(f"--- softmax: x - torch.max(x) ---", file=log_file)
        print(f"x shape: {x.shape}", file=log_file)
        print(f"x max: {x.max()}", file=log_file)
        print(f"x min: {x.min()}", file=log_file)
        print(f"x: {x}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    two_x = torch.where(
            x>=-4,
            torch.pow(2.0, x),
            0
        )

    if log_file is not None:
        print(f"--- softmax: 2^x ---", file=log_file)
        print(f"two_x shape: {two_x.shape}", file=log_file)
        print(f"two_x max: {two_x.max()}", file=log_file)
        print(f"two_x min: {two_x.min()}", file=log_file)
        print(f"two_x: {two_x}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    sum_two_x = torch.sum(two_x, dim=dim, keepdim=True)

    if log_file is not None:
        print(f"--- softmax: sum(2^x) ---", file=log_file)
        print(f"sum_two_x shape: {sum_two_x.shape}", file=log_file)
        print(f"sum_two_x max: {sum_two_x.max()}", file=log_file)
        print(f"sum_two_x min: {sum_two_x.min()}", file=log_file)
        print(f"sum_two_x: {sum_two_x}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    output = two_x / sum_two_x

    if log_file is not None:
        print(f"--- softmax: 2^x / sum(2^x) ---", file=log_file)
        print(f"output shape: {output.shape}", file=log_file)
        print(f"output max: {output.max()}", file=log_file)
        print(f"output min: {output.min()}", file=log_file)
        print(f"output: {output}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    return output

class FloatAttnMaskSoftmax(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.num_heads = num_heads

    def forward(self, attn, mask, B_, N):
        if mask is not None:
            nW = mask.shape[1]

            if log_file is not None:
                print(f"--- mask ---", file=log_file)
                print(f"mask shape: {mask.shape}", file=log_file)
                print(f"mask: {mask}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            attn = attn.view(B_ // nW, nW, 1, N, N)

            if log_file is not None:
                print(f"--- attn.view(B_ // nW, nW, 1, N, N) ---", file=log_file)
                print(f"attn shape: {attn.view(B_ // nW, nW, 1, N, N).shape}", file=log_file)
                print(f"attn max: {attn.view(B_ // nW, nW, 1, N, N).max()}", file=log_file)
                print(f"attn min: {attn.view(B_ // nW, nW, 1, N, N).min()}", file=log_file)
                print(f"attn: {attn.view(B_ // nW, nW, 1, N, N)}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            attn = attn + mask

            if log_file is not None:
                print(f"--- attn + mask ---", file=log_file)
                print(f"attn shape: {attn.shape}", file=log_file)
                print(f"attn max: {attn.max()}", file=log_file)
                print(f"attn min: {attn.min()}", file=log_file)
                print(f"attn: {attn}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            attn = attn.view(-1, 1, N, N)

            if log_file is not None:
                print(f"--- attn.view(-1, 1, N, N) ---", file=log_file)
                print(f"attn shape: {attn.shape}", file=log_file)
                print(f"attn max: {attn.max()}", file=log_file)
                print(f"attn min: {attn.min()}", file=log_file)
                print(f"attn: {attn}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

        # base-2 softmax via ln(2)
        attn = softmax(attn)
        return attn

In [5]:
def conv2d(x_int8, weight_int8, bias_int32, stride=1):

    batch_size = x_int8.shape[0]

    outputs = []
    for b in range(batch_size):
        res = torch._int_mm(weight_int8, x_int8[b])
        if bias_int32 is not None:
            res += bias_int32
        outputs.append(res)

    output_tensor = torch.stack(outputs).view(batch_size, 96, 56, 56)

    return output_tensor

def linear(x_int8, weight_int8, bias_int32=None):
    original_shape = x_int8.shape
    x_i8 = x_int8.reshape(-1, original_shape[-1]).to(torch.int8)
    w_i8 = weight_int8.to(torch.int8).t()

    output_matmul = torch._int_mm(x_i8, w_i8)

    if log_file is not None:
        print(f"--- x @ w^t ---", file=log_file)
        print(f"x @ w^t shape: {output_matmul.shape}", file=log_file)
        print(f"x @ w^t max: {output_matmul.max()}", file=log_file)
        print(f"x @ w^t min: {output_matmul.min()}", file=log_file)
        print(f"x @ w^t: {output_matmul}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    if bias_int32 is not None:
        output_matmul = output_matmul + bias_int32

        if log_file is not None:
            print(f"--- x @ w^t + b ---", file=log_file)
            print(f"x @ w^t + b shape: {output_matmul.shape}", file=log_file)
            print(f"x @ w^t + b max: {output_matmul.max()}", file=log_file)
            print(f"x @ w^t + b min: {output_matmul.min()}", file=log_file)
            print(f"x @ w^t + b: {output_matmul}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

    new_shape = list(original_shape[:-1]) + [weight_int8.shape[0]]

    final_output = output_matmul.reshape(new_shape)

    return final_output

In [6]:
def matmul_int8_4d(x_int8, y_int8):

    B, H, N, K = x_int8.shape
    D = y_int8.shape[-1]

    x_flat = x_int8.reshape(-1, N, K).to(torch.int8)
    y_flat = y_int8.reshape(-1, K, D).to(torch.int8)

    D_padded = (D + 7) // 8 * 8
    pad_D = D_padded - D

    K_padded = (K + 7) // 8 * 8
    pad_K = K_padded - K

    if pad_D > 0 or pad_K > 0:
        x_flat = torch.nn.functional.pad(x_flat, (0, pad_K, 0, 0))
        y_flat = torch.nn.functional.pad(y_flat, (0, pad_D, 0, pad_K))

    results = []
    for i in range(B * H):
        res = torch._int_mm(x_flat[i], y_flat[i])

        if pad_D > 0:
            res = res[:, :D]
        results.append(res)

    output = torch.stack(results).view(B, H, N, D)
    return output

In [7]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None,
                 out_features=None, act_layer=nn.ReLU):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        # self.fuse_linear_bn1 = nn.Linear(in_features, hidden_features)
        # self.fuse_linear_bn2 = nn.Linear(hidden_features, out_features)

        self.register_buffer("fuse_linear_bn1_weights", None)
        self.register_buffer("fuse_linear_bn1_bias", None)
        self.register_buffer("fuse_linear_bn1_output_scale", None)

        self.register_buffer("fuse_linear_bn2_weights", None)
        self.register_buffer("fuse_linear_bn2_bias", None)
        self.register_buffer("fuse_linear_bn2_output_scale", None)

    def forward(self, x):

        # self.fuse_linear_bn1_weights = self.fuse_linear_bn1.weight.to(torch.int8)

        if log_file is not None:
            print(f"--- Mlp fuse_linear_bn1 weights int ---", file=log_file)
            print(f"w_int8 shape: {self.fuse_linear_bn1_weights.shape}", file=log_file)
            print(f"w_int8 max: {self.fuse_linear_bn1_weights.max()}", file=log_file)
            print(f"w_int8 min: {self.fuse_linear_bn1_weights.min()}", file=log_file)
            print(f"w_int8: {self.fuse_linear_bn1_weights}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # self.fuse_linear_bn1_bias = self.fuse_linear_bn1.bias.to(torch.int32)

        if log_file is not None:
            print(f"--- Mlp fuse_linear_bn1 bias int ---", file=log_file)
            print(f"b_int32 shape: {self.fuse_linear_bn1_bias.shape}", file=log_file)
            print(f"b_int32 max: {self.fuse_linear_bn1_bias.max()}", file=log_file)
            print(f"b_int32 min: {self.fuse_linear_bn1_bias.min()}", file=log_file)
            print(f"b_int32: {self.fuse_linear_bn1_bias}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = linear(x, self.fuse_linear_bn1_weights, self.fuse_linear_bn1_bias)

        if log_file is not None:
            print(f"--- Mlp fuse_linear_bn1 linear ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = quantize(x, self.fuse_linear_bn1_output_scale, is_input=False)

        x = nn.ReLU()(x)

        if log_file is not None:
            print(f"--- Mlp ReLU ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # self.fuse_linear_bn2_weights = self.fuse_linear_bn2.weight.to(torch.int8)

        if log_file is not None:
            print(f"--- Mlp fuse_linear_bn2 weights int ---", file=log_file)
            print(f"w_int8 shape: {self.fuse_linear_bn2_weights.shape}", file=log_file)
            print(f"w_int8 max: {self.fuse_linear_bn2_weights.max()}", file=log_file)
            print(f"w_int8 min: {self.fuse_linear_bn2_weights.min()}", file=log_file)
            print(f"w_int8: {self.fuse_linear_bn2_weights}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # self.fuse_linear_bn2_bias = self.fuse_linear_bn2.bias.to(torch.int32)

        if log_file is not None:
            print(f"--- Mlp fuse_linear_bn2 bias int ---", file=log_file)
            print(f"b_int32 shape: {self.fuse_linear_bn2_bias.shape}", file=log_file)
            print(f"b_int32 max: {self.fuse_linear_bn2_bias.max()}", file=log_file)
            print(f"b_int32 min: {self.fuse_linear_bn2_bias.min()}", file=log_file)
            print(f"b_int32: {self.fuse_linear_bn2_bias}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = linear(x, self.fuse_linear_bn2_weights, self.fuse_linear_bn2_bias)

        if log_file is not None:
            print(f"--- Mlp fuse_linear_bn2 linear ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = quantize(x, self.fuse_linear_bn2_output_scale, is_input=False)

        return x

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)

    if log_file is not None:
        print(f"--- window_partition: x.view(B, H // window_size, window_size, W // window_size, window_size, C) ---", file=log_file)
        print(f"window_size: {window_size}", file=log_file)
        print(f"x shape: {x.shape}", file=log_file)
        print(f"x max: {x.max()}", file=log_file)
        print(f"x min: {x.min()}", file=log_file)
        print(f"x: {x}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)

    if log_file is not None:
        print(f"--- window_partition: windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C) ---", file=log_file)
        print(f"window_size: {window_size}", file=log_file)
        print(f"windows shape: {windows.shape}", file=log_file)
        print(f"windows max: {windows.max()}", file=log_file)
        print(f"windows min: {windows.min()}", file=log_file)
        print(f"windows: {windows}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    return windows


def window_reverse(windows, window_size, H, W):
    B = windows.shape[0] // (H * W // window_size // window_size)

    if log_file is not None:
        print(f"--- window_reverse: B = windows.shape[0] // (H * W // window_size // window_size) ---", file=log_file)
        print(f"window_size: {window_size}", file=log_file)
        print(f"B: {B}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)

    if log_file is not None:
        print(f"--- window_reverse: x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1) ---", file=log_file)
        print(f"window_size: {window_size}", file=log_file)
        print(f"x shape: {x.shape}", file=log_file)
        print(f"x max: {x.max()}", file=log_file)
        print(f"x min: {x.min()}", file=log_file)
        print(f"x: {x}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)

    if log_file is not None:
        print(f"--- window_reverse: x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1) ---", file=log_file)
        print(f"x shape: {x.shape}", file=log_file)
        print(f"x max: {x.max()}", file=log_file)
        print(f"x min: {x.min()}", file=log_file)
        print(f"x: {x}", file=log_file)
        print("-" * 40, file=log_file)
        log_file.flush()

    return x


class WindowAttention(nn.Module):

    def __init__(self, dim, window_size, num_heads, qkv_bias=True):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        # self.scale = (0.5 if num_heads == 3 else
        #              0.5 if num_heads == 6 else
        #              0.25 if num_heads == 12 else
        #              0.25 if num_heads == 24 else
        #              head_dim ** -0.5)

        # self.scale = torch.tensor(self.scale)

        # self.fuse_qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)

        # self.proj = nn.Linear(dim, dim)

        self.float_mask_softmax = FloatAttnMaskSoftmax(num_heads)

        self.register_buffer("q_weight", None)
        self.register_buffer("q_bias", None)
        self.register_buffer("q_output_scale", None)

        self.register_buffer("k_weight", None)
        self.register_buffer("k_bias", None)
        self.register_buffer("k_output_scale", None)

        self.register_buffer("relative_position_bias", None)
        self.register_buffer("qk_output_scale", None)


        self.register_buffer("v_weight", None)
        self.register_buffer("v_bias", None)
        self.register_buffer("v_output_scale", None)

        self.register_buffer("attn_v_output_scale", None)

        self.register_buffer("proj_weights", None)
        self.register_buffer("proj_bias", None)
        self.register_buffer("proj_output_scale", None)

    def forward(self, x, mask):

        B_, N, C = x.shape

        # ── Q / K / V linear projections ────────────────────────────────────
        if log_file is not None:
            for name, w, b in [("q", self.q_weight, self.q_bias),
                                ("k", self.k_weight, self.k_bias),
                                ("v", self.v_weight, self.v_bias)]:
                print(f"--- {name}_weight ---", file=log_file)
                print(f"{name}_weight shape: {w.shape}", file=log_file)
                print(f"{name}_weight max: {w.max()}", file=log_file)
                print(f"{name}_weight min: {w.min()}", file=log_file)
                print(f"{name}_weight: {w}", file=log_file)
                print("-" * 40, file=log_file)
                print(f"--- {name}_bias ---", file=log_file)
                print(f"{name}_bias shape: {b.shape}", file=log_file)
                print(f"{name}_bias max: {b.max()}", file=log_file)
                print(f"{name}_bias min: {b.min()}", file=log_file)
                print(f"{name}_bias: {b}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

        q = linear(x, self.q_weight, self.q_bias)
        k = linear(x, self.k_weight, self.k_bias)
        v = linear(x, self.v_weight, self.v_bias)

        if log_file is not None:
            for name, t in [("q", q), ("k", k), ("v", v)]:
                print(f"--- WindowAttention {name} linear ---", file=log_file)
                print(f"{name} shape: {t.shape}", file=log_file)
                print(f"{name} max: {t.max()}", file=log_file)
                print(f"{name} min: {t.min()}", file=log_file)
                print(f"{name}: {t}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

        # ── Reshape to (B_, num_heads, N, head_dim) ──────────────────────────
        q = q.reshape(B_, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        k = k.reshape(B_, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        v = v.reshape(B_, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        if log_file is not None:
            for name, t in [("q", q), ("k", k), ("v", v)]:
                print(f"--- {name}.reshape(B_, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3) ---", file=log_file)
                print(f"{name} shape: {t.shape}", file=log_file)
                print(f"{name} max: {t.max()}", file=log_file)
                print(f"{name} min: {t.min()}", file=log_file)
                print(f"{name}: {t}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

        # ── Split into per-head variables ─────────────────────────────────────
        q_heads = [q[:, i, :, :] for i in range(self.num_heads)]
        k_heads = [k[:, i, :, :] for i in range(self.num_heads)]
        v_heads = [v[:, i, :, :] for i in range(self.num_heads)]

        head_dim = C // self.num_heads

        out_heads = []
        for i in range(self.num_heads):

            # ── Quantize Q / K for this head ──────────────────────────────────
            q_i = quantize(q_heads[i], self.q_output_scale[i * head_dim : (i+1) * head_dim], is_input=False).unsqueeze(1)
            k_i = quantize(k_heads[i], self.k_output_scale[i * head_dim : (i+1) * head_dim], is_input=False).unsqueeze(1)

            # ── Attention scores: Q @ K^T ─────────────────────────────────────
            if log_file is not None:
                print(f"--- [head {i}] k.transpose(-2, -1) ---", file=log_file)
                kt = k_i.transpose(-2, -1)
                print(f"k shape: {kt.shape}", file=log_file)
                print(f"k max: {kt.max()}", file=log_file)
                print(f"k min: {kt.min()}", file=log_file)
                print(f"k: {kt}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            attn_i = matmul_int8_4d(q_i, k_i.transpose(-2, -1))

            if log_file is not None:
                print(f"--- [head {i}] q @ k^t ---", file=log_file)
                print(f"q @ k^t shape: {attn_i.shape}", file=log_file)
                print(f"q @ k^t max: {attn_i.max()}", file=log_file)
                print(f"q @ k^t min: {attn_i.min()}", file=log_file)
                print(f"q @ k^t: {attn_i}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            # ── Relative position bias ────────────────────────────────────────
            if log_file is not None:
                rpb = self.relative_position_bias[:, i:i+1, :, :]
                print(f"--- [head {i}] relative_position_bias ---", file=log_file)
                print(f"relative_position_bias shape: {rpb.shape}", file=log_file)
                print(f"relative_position_bias max: {rpb.max()}", file=log_file)
                print(f"relative_position_bias min: {rpb.min()}", file=log_file)
                print(f"relative_position_bias: {rpb}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            attn_i = attn_i + self.relative_position_bias[:, i:i+1, :, :]

            if log_file is not None:
                print(f"--- [head {i}] attn + relative_position_bias ---", file=log_file)
                print(f"attn + relative_position_bias shape: {attn_i.shape}", file=log_file)
                print(f"attn + relative_position_bias max: {attn_i.max()}", file=log_file)
                print(f"attn + relative_position_bias min: {attn_i.min()}", file=log_file)
                print(f"attn + relative_position_bias: {attn_i}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            attn_i = quantize(attn_i, self.qk_output_scale, is_input=False)
            attn_i = self.float_mask_softmax(attn_i, mask, B_, N)
            attn_i = log2_quantize(attn_i)

            # ── Quantize V and compute attention output ────────────────────────
            v_i = quantize(v_heads[i], self.v_output_scale[i * head_dim : (i+1) * head_dim], is_input=False).unsqueeze(1)

            out_i = matmul_int8_4d(attn_i, v_i)

            if log_file is not None:
                print(f"--- [head {i}] attn @ v ---", file=log_file)
                print(f"attn @ v shape: {out_i.shape}", file=log_file)
                print(f"attn @ v max: {out_i.max()}", file=log_file)
                print(f"attn @ v min: {out_i.min()}", file=log_file)
                print(f"attn @ v: {out_i}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            out_i = quantize(out_i, self.attn_v_output_scale, is_input=False)
            out_heads.append(out_i)

        # ── Recombine heads ───────────────────────────────────────────────────
        x = torch.cat(out_heads, dim=1)           # (B_, num_heads, N, head_dim)
        x = x.transpose(1, 2).reshape(B_, N, C)   # (B_, N, C)

        if log_file is not None:
            print(f"--- x.transpose(1, 2).reshape(B_, N, C) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # ── Projection ────────────────────────────────────────────────────────
        if log_file is not None:
            print(f"--- WindowAttention proj weights int ---", file=log_file)
            print(f"w_int8 shape: {self.proj_weights.shape}", file=log_file)
            print(f"w_int8 max: {self.proj_weights.max()}", file=log_file)
            print(f"w_int8 min: {self.proj_weights.min()}", file=log_file)
            print(f"w_int8: {self.proj_weights}", file=log_file)
            print("-" * 40, file=log_file)
            print(f"--- WindowAttention proj bias int ---", file=log_file)
            print(f"b_int32 shape: {self.proj_bias.shape}", file=log_file)
            print(f"b_int32 max: {self.proj_bias.max()}", file=log_file)
            print(f"b_int32 min: {self.proj_bias.min()}", file=log_file)
            print(f"b_int32: {self.proj_bias}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = linear(x, self.proj_weights, self.proj_bias)

        if log_file is not None:
            print(f"--- WindowAttention proj linear ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = quantize(x, self.proj_output_scale, is_input=False)

        return x

class SwinTransformerBlock(nn.Module):

    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, act_layer=nn.ReLU):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias)

        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer)

        self.register_buffer("attn_mask", None)

        # self.input_norm1 = nn.Linear(dim, dim)
        # self.input_norm2 = nn.Linear(dim, dim)

        # self.register_buffer("input_norm1_output_shift", None)
        # self.register_buffer("shortcut_add1_shift", None)
        # self.register_buffer("input_norm2_output_shift", None)
        # self.register_buffer("x_add2_shift", None)

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x

        # w_int8 = self.input_norm1.weight.to(torch.int8)

        # if log_file is not None:
        #     print(f"--- SwinTransformerBlock input_norm1 weight int ---", file=log_file)
        #     print(f"w_int8 shape: {w_int8.shape}", file=log_file)
        #     print(f"w_int8 max: {w_int8.max()}", file=log_file)
        #     print(f"w_int8 min: {w_int8.min()}", file=log_file)
        #     print(f"w_int8: {w_int8}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # b_int32 = self.input_norm1.bias.to(torch.int32)

        # if log_file is not None:
        #     print(f"--- SwinTransformerBlock input_norm1 bias int ---", file=log_file)
        #     print(f"b_int32 shape: {b_int32.shape}", file=log_file)
        #     print(f"b_int32 max: {b_int32.max()}", file=log_file)
        #     print(f"b_int32 min: {b_int32.min()}", file=log_file)
        #     print(f"b_int32: {b_int32}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # x = linear(x, w_int8, b_int32)

        # if log_file is not None:
        #     print(f"--- SwinTransformerBlock input_norm1 ---", file=log_file)
        #     print(f"x shape: {x.shape}", file=log_file)
        #     print(f"x max: {x.max()}", file=log_file)
        #     print(f"x min: {x.min()}", file=log_file)
        #     print(f"x: {x}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # x = quantize(x, self.input_norm1_output_shift, is_input=False)

        x = x.view(B, H, W, C)

        if log_file is not None:
            print(f"--- x.view(B, H, W, C) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

            if log_file is not None:
                print(f"--- shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2)) ---", file=log_file)
                print(f"shift_size: {self.shift_size}", file=log_file)
                print(f"shifted_x shape: {shifted_x.shape}", file=log_file)
                print(f"shifted_x max: {shifted_x.max()}", file=log_file)
                print(f"shifted_x min: {shifted_x.min()}", file=log_file)
                print(f"shifted_x: {shifted_x}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
        else:
            shifted_x = x
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C

        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C

        if log_file is not None:
            print(f"--- x_windows = x_windows.view(-1, self.window_size * self.window_size, C) ---", file=log_file)
            print(f"window_size: {self.window_size}", file=log_file)
            print(f"x_windows shape: {x_windows.shape}", file=log_file)
            print(f"x_windows max: {x_windows.max()}", file=log_file)
            print(f"x_windows min: {x_windows.min()}", file=log_file)
            print(f"x_windows: {x_windows}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # W-MSA/SW-MSA
        # self.attn_mask = self.attn_mask.unsqueeze(1).unsqueeze(0) if self.attn_mask is not None else None

        attn_windows = self.attn(x_windows, self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)

        if log_file is not None:
            print(f"--- attn_windows.view(-1, self.window_size, self.window_size, C) ---", file=log_file)
            print(f"window_size: {self.window_size}", file=log_file)
            print(f"attn_windows shape: {attn_windows.shape}", file=log_file)
            print(f"attn_windows max: {attn_windows.max()}", file=log_file)
            print(f"attn_windows min: {attn_windows.min()}", file=log_file)
            print(f"attn_windows: {x_windows}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # reverse cyclic shift
        if self.shift_size > 0:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

            if log_file is not None:
                print(f"--- x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2)) ---", file=log_file)
                print(f"shift_size: {self.shift_size}", file=log_file)
                print(f"x shape: {x.shape}", file=log_file)
                print(f"x max: {x.max()}", file=log_file)
                print(f"x min: {x.min()}", file=log_file)
                print(f"x: {x}", file=log_file)
                print("-" * 40, file=log_file)
                log_file.flush()

        else:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = shifted_x

        x = x.view(B, H * W, C)

        if log_file is not None:
            print(f"--- x = x.view(B, H * W, C) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # shortcut = quantize(shortcut, self.shortcut_add1_shift, is_input=False)
        x = shortcut + x

        if log_file is not None:
            print(f"--- x = shortcut + x ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # FFN
        # w_int8 = self.input_norm2.weight.to(torch.int8)

        # if log_file is not None:
        #     print(f"--- SwinTransformerBlock input_norm2 weight int ---", file=log_file)
        #     print(f"w_int8 shape: {w_int8.shape}", file=log_file)
        #     print(f"w_int8 max: {w_int8.max()}", file=log_file)
        #     print(f"w_int8 min: {w_int8.min()}", file=log_file)
        #     print(f"w_int8: {w_int8}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # b_int32 = self.input_norm2.bias.to(torch.int32)

        # if log_file is not None:
        #     print(f"--- SwinTransformerBlock input_norm2 bias int ---", file=log_file)
        #     print(f"b_int32 shape: {b_int32.shape}", file=log_file)
        #     print(f"b_int32 max: {b_int32.max()}", file=log_file)
        #     print(f"b_int32 min: {b_int32.min()}", file=log_file)
        #     print(f"b_int32: {b_int32}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # x = linear(x, w_int8, b_int32)

        # if log_file is not None:
        #     print(f"--- SwinTransformerBlock input_norm2 ---", file=log_file)
        #     print(f"x shape: {x.shape}", file=log_file)
        #     print(f"x max: {x.max()}", file=log_file)
        #     print(f"x min: {x.min()}", file=log_file)
        #     print(f"x: {x}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # x = quantize(x, self.input_norm2_output_shift, is_input=False)

        x_mlp = self.mlp(x)

        # x = quantize(x, self.x_add2_shift, is_input=False)
        x = x + x_mlp

        if log_file is not None:
            print(f"--- x = x + x_mlp ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        return x


class PatchMerging(nn.Module):

    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim

        # self.input_norm = nn.Linear(4 * dim, 2 * dim, bias=True)
        # self.input_norm = nn.Linear(4 * dim, 4 * dim)

        self.register_buffer("input_norm_weights", None)
        self.register_buffer("input_norm_bias", None)

        # self.register_buffer("reduction_output_shift", None)
        self.register_buffer("input_norm_output_scale", None)

    def forward(self, x):

        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."


        x = x.view(B, H, W, C)

        if log_file is not None:
            print(f"--- x.view(B, H, W, C) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C

        if log_file is not None:
            print(f"--- x[:, 0::2, 0::2, :] # B H/2 W/2 C ---", file=log_file)
            print(f"x0 shape: {x0.shape}", file=log_file)
            print(f"x0 max: {x0.max()}", file=log_file)
            print(f"x0 min: {x0.min()}", file=log_file)
            print(f"x0: {x0}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C

        if log_file is not None:
            print(f"--- x[:, 1::2, 0::2, :] # B H/2 W/2 C ---", file=log_file)
            print(f"x1 shape: {x1.shape}", file=log_file)
            print(f"x1 max: {x1.max()}", file=log_file)
            print(f"x1 min: {x1.min()}", file=log_file)
            print(f"x1: {x1}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C

        if log_file is not None:
            print(f"--- x[:, 0::2, 1::2, :] # B H/2 W/2 C ---", file=log_file)
            print(f"x2 shape: {x2.shape}", file=log_file)
            print(f"x2 max: {x2.max()}", file=log_file)
            print(f"x2 min: {x2.min()}", file=log_file)
            print(f"x2: {x2}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C

        if log_file is not None:
            print(f"--- x[:, 1::2, 1::2, :] # B H/2 W/2 C ---", file=log_file)
            print(f"x3 shape: {x3.shape}", file=log_file)
            print(f"x3 max: {x3.max()}", file=log_file)
            print(f"x3 min: {x3.min()}", file=log_file)
            print(f"x3: {x3}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C

        if log_file is not None:
            print(f"--- x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        if log_file is not None:
            print(f"--- x.view(B, -1, 4 * C)  # B H/2*W/2 4*C ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # self.input_norm_weights = self.input_norm.weight.to(torch.int8)

        if log_file is not None:
            print(f"--- PatchMerging input_norm weight int ---", file=log_file)
            print(f"w_int8 shape: {self.input_norm_weights.shape}", file=log_file)
            print(f"w_int8 max: {self.input_norm_weights.max()}", file=log_file)
            print(f"w_int8 min: {self.input_norm_weights.min()}", file=log_file)
            print(f"w_int8: {self.input_norm_weights}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # self.input_norm_bias = self.input_norm.bias.to(torch.int32)

        if log_file is not None:
            print(f"--- PatchMerging input_norm bias int ---", file=log_file)
            print(f"b_int32 shape: {self.input_norm_bias.shape}", file=log_file)
            print(f"b_int32 max: {self.input_norm_bias.max()}", file=log_file)
            print(f"b_int32 min: {self.input_norm_bias.min()}", file=log_file)
            print(f"b_int32: {self.input_norm_bias}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = linear(x, self.input_norm_weights, self.input_norm_bias)

        if log_file is not None:
            print(f"--- PatchMerging input_norm linear ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = quantize(x, self.input_norm_output_scale, is_input=False)

        # w_int8 = self.reduction.weight.to(torch.int8)

        # if log_file is not None:
        #     print(f"--- PatchMerging reduction weight int ---", file=log_file)
        #     print(f"w_int8 shape: {w_int8.shape}", file=log_file)
        #     print(f"w_int8 max: {w_int8.max()}", file=log_file)
        #     print(f"w_int8 min: {w_int8.min()}", file=log_file)
        #     print(f"w_int8: {w_int8}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # x = linear(x, w_int8)

        # if log_file is not None:
        #     print(f"--- PatchMerging reduction linear ---", file=log_file)
        #     print(f"x shape: {x.shape}", file=log_file)
        #     print(f"x max: {x.max()}", file=log_file)
        #     print(f"x min: {x.min()}", file=log_file)
        #     print(f"x: {x}", file=log_file)
        #     print("-" * 40, file=log_file)
        #     log_file.flush()

        # x = quantize(x, self.reduction_output_shift, is_input=False)

        return x


class BasicLayer(nn.Module):

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, downsample=None):

        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x


class PatchEmbed(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        # self.fused_proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

        self.register_buffer("fused_proj_weights", None)
        self.register_buffer("fused_proj_bias", None)

        self.register_buffer("fused_proj_output_scale", None)

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."

        # self.fused_proj_weights = self.fused_proj.weight.to(torch.int8)

        # 96 * 48
        w_int8 = self.fused_proj_weights.view(self.embed_dim, -1)

        if log_file is not None:
            print(f"--- fused_proj weights int ---", file=log_file)
            print(f"w_int8 shape: {w_int8.shape}", file=log_file)
            print(f"w_int8 max: {w_int8.max()}", file=log_file)
            print(f"w_int8 min: {w_int8.min()}", file=log_file)
            print(f"w_int8: {w_int8}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # 96 * 1
        # self.fused_proj_bias = self.fused_proj.bias.to(torch.int32)#.view(-1, 1)
        b_int32 = self.fused_proj_bias.view(-1, 1)

        if log_file is not None:
            print(f"--- fused_proj bias int ---", file=log_file)
            print(f"b_int32 shape: {b_int32.shape}", file=log_file)
            print(f"b_int32 max: {b_int32.max()}", file=log_file)
            print(f"b_int32 min: {b_int32.min()}", file=log_file)
            print(f"b_int32: {b_int32}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # 3 * 224 * 224 -> 48 * 3136
        x = torch.nn.functional.unfold(x.float(), kernel_size=(self.patch_size[0], self.patch_size[1]), stride=self.patch_size[0]).to(torch.int8)#software

        if log_file is not None:
            print(f"--- patchembed x = torch.nn.functional.unfold(x, kernel_size=(self.patch_size[0], self.patch_size[1]), stride=self.patch_size[0]) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # 96 * 3136
        x = conv2d(x, w_int8, b_int32, stride=self.patch_size[0])

        if log_file is not None:
            print(f"--- patchembed conv2d ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # 96 * 56 * 56 -> 96 * 3136 -> 3136 * 96
        x = x.flatten(2).transpose(1, 2)

        if log_file is not None:
            print(f"--- x.transpose(1, 2) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = quantize(x, self.fused_proj_output_scale, is_input=False)

        return x

def avg_pool(x):
    length = x.shape[2] # 49
    sum_x = torch.sum(x, dim=2, keepdim=True)

    if log_file is not None:
            print(f"--- sum dim 2 of x (49) ---", file=log_file)
            print(f"sum_x shape: {sum_x.shape}", file=log_file)
            print(f"sum_x: {sum_x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

    output = sum_x / length

    if log_file is not None:
            print(f"--- (sum dim 2 of x) / length (49) ---", file=log_file)
            print(f"output shape: {output.shape}", file=log_file)
            print(f"output: {output}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

    return output


class SwinTransformer(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True, **kwargs):
        super().__init__()

        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                               input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                                 patches_resolution[1] // (2 ** i_layer)),
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               mlp_ratio=self.mlp_ratio,
                               qkv_bias=qkv_bias,
                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None)
            self.layers.append(layer)

        # self.avgpool = nn.AdaptiveAvgPool1d(1)


        # self.input_norm = nn.Linear(self.num_features, self.num_features)
        self.register_buffer("input_scale", None)

        self.register_buffer("input_norm_weights", None)
        self.register_buffer("input_norm_bias", None)
        self.register_buffer("input_norm_output_scale", None)
        # self.register_buffer("head_weights", None)
        # self.register_buffer("head_bias", None)

        self.head = nn.Sequential(OrderedDict([("fc", nn.Linear(self.num_features, num_classes))])) if num_classes > 0 else nn.Identity()
        self.register_buffer("output_scale", None)

    def forward_features(self, x):

        if log_file is not None:
            print(f"--- input (x) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()
        # 3 * 224 * 224
        x = quantize(x, self.input_scale)#software

        if log_file is not None:
            print(f"--- float to int ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = self.patch_embed(x)

        for layer in self.layers:
            x = layer(x)

        # self.input_norm_weights = self.input_norm.weight.to(torch.int8)

        if log_file is not None:
            print("--- SwinTransformer input_norm weight int ---", file=log_file)
            print(f"w_int8 shape: {self.input_norm_weights.shape}", file=log_file)
            print(f"w_int8 max: {self.input_norm_weights.max()}", file=log_file)
            print(f"w_int8 min: {self.input_norm_weights.min()}", file=log_file)
            print(f"w_int8: {self.input_norm_weights}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        # self.input_norm_bias = self.input_norm.bias.to(torch.int32)

        if log_file is not None:
            print("--- SwinTransformer input_norm bias int ---", file=log_file)
            print(f"b_int32 shape: {self.input_norm_bias.shape}", file=log_file)
            print(f"b_int32 max: {self.input_norm_bias.max()}", file=log_file)
            print(f"b_int32 min: {self.input_norm_bias.min()}", file=log_file)
            print(f"b_int32: {self.input_norm_bias}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = linear(x, self.input_norm_weights, self.input_norm_bias)

        if log_file is not None:
            print("--- SwinTransformer input_norm linear ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x max: {x.max()}", file=log_file)
            print(f"x min: {x.min()}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = quantize(x, self.input_norm_output_scale, is_input=False)

        x = x.to(torch.float32)

        x = x.transpose(1, 2)#software

        if log_file is not None:
            print("--- x.transpose(1, 2) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = avg_pool(x)#software  # B C 1

        if log_file is not None:
            print("--- SwinTransformer avgpool ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = torch.flatten(x, 1)#software

        if log_file is not None:
            print("--- x = torch.flatten(x, 1) ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        return x

    def forward(self, x):
        x = self.forward_features(x)

        x = self.head(x)#software

        if log_file is not None:
            print("--- SwinTransformer head ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        x = dequantize(x, self.output_scale)#software

        if log_file is not None:
            print("--- dequantize ---", file=log_file)
            print(f"x shape: {x.shape}", file=log_file)
            print(f"x: {x}", file=log_file)
            print("-" * 40, file=log_file)
            log_file.flush()

        return x

In [17]:
# 2. Create your custom implementation
custom_model = SwinTransformer(
    img_size=224,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[3, 6, 12, 24],
)

In [9]:
orig_state = torch.load("/content/drive/MyDrive/final_weights15.pth")

In [18]:
for name, param in orig_state.items():
    # Check if the name corresponds to an observer's scale
    if "scale" in name or "relative_position_bias" in name or "shift" in name or "attn_mask" in name or "_weight" in name or "_bias" in name and "table" not in name:
        # Construct the path to the observer module
        parts = name.split('.')
        current_module = custom_model
        module_found = True
        for i, part in enumerate(parts[:-1]):
            if hasattr(current_module, part):
                current_module = getattr(current_module, part)
            else:
                # If a part of the path doesn't exist, this key is not applicable
                module_found = False
                break

        if module_found:
            attr_name = parts[-1] # This should be 'scale'
            if hasattr(current_module, attr_name):
                # Directly assign the parameter from the loaded state to the model's buffer
                # This will resize the buffer if necessary, fixing the shape mismatch.
                with torch.no_grad(): # Ensure this operation doesn't track gradients
                    # Ensure device consistency by moving param to buffer's device if different
                    setattr(current_module, attr_name, param)
            else:
                print(f"Warning: Attribute '{attr_name}' not found in module '{'.'.join(parts[:-1])}' for key '{name}'. Skipping.")
        else:
            print(f"Warning: Module path for key '{name}' not fully resolvable. Skipping.")


# Load into your model
missing, unexpected = custom_model.load_state_dict(orig_state, strict=False)


# Optionally inspect for any mismatches
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: []
Unexpected keys: []


In [11]:
!hf auth login

A new version of huggingface_hub (1.5.0) is available! You are using version 1.4.1.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: 
Token is valid (permission: fineGrained).
The token `swin` has been saved to /root/.cache/huggingface/sto

In [21]:
# ----------------------------------------------------------------------------
# 1) Load ImageNet-1k training set (streaming mode)
# ----------------------------------------------------------------------------
hf_dataset_train = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True)
hf_dataset_val = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)


train_stream = hf_dataset_train.take(100_000)
val_stream   = hf_dataset_train.skip(100_000).take(15_000)
test_stream = hf_dataset_val.take(10_000)


# ----------------------------------------------------------------------------
# 2) Preprocessing pipeline
# ----------------------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def collate_fn(batch):
    """Convert HF samples into tensors"""
    images, labels = [], []
    for sample in batch:
        img = sample["image"]
        if not isinstance(img, Image.Image):  # if numpy array
            img = Image.fromarray(img)
        img = img.convert("RGB")  # <-- ensure 3 channels
        images.append(transform(img))
        labels.append(sample["label"])
    return torch.stack(images), torch.tensor(labels)


# ----------------------------------------------------------------------------
# 3) Wrap datasets with DataLoader
# ----------------------------------------------------------------------------
train_loader = DataLoader(train_stream, batch_size=1, collate_fn=collate_fn)
val_loader   = DataLoader(val_stream,   batch_size=64, collate_fn=collate_fn)
test_loader  = DataLoader(test_stream, batch_size=64, collate_fn=collate_fn)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    top1, top5, total, total_loss = 0, 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(dataloader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Top-1
            _, pred1 = outputs.topk(1, dim=1)
            top1 += (pred1.squeeze() == labels).sum().item()

            # Top-5
            _, pred5 = outputs.topk(5, dim=1)
            top5 += sum([labels[j].item() in pred5[j] for j in range(labels.size(0))])

            total += labels.size(0)

            if (i+1) % 1 == 0:
                print(f"[Val Step {i+1}] "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Top-1: {100.*top1/total:.2f}% | "
                      f"Top-5: {100.*top5/total:.2f}%")

            break

    return 100.*top1/total, 100.*top5/total, total_loss/(i+1)

# ----------------------------------------------------------------------------
# 5) Train the model
# ----------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
custom_model.to(device)

criterion = nn.CrossEntropyLoss()

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [22]:
log_file = open("model_logs.txt", "w")
# log_file = None

In [23]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, train_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 0.0000 | Top-1: 100.00% | Top-5: 100.00%

PTQ Model Results:
Top-1: 100.00% | Top-5: 100.00%


In [24]:
log_file.close()

drive_path = "/content/drive/MyDrive/model_logs.txt"
shutil.copy("model_logs.txt", drive_path)

'/content/drive/MyDrive/model_logs.txt'

In [ ]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, test_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 1.0925 | Top-1: 78.12% | Top-5: 89.06%
[Val Step 2] Loss: 1.2648 | Top-1: 73.44% | Top-5: 86.72%
[Val Step 3] Loss: 1.2471 | Top-1: 73.44% | Top-5: 87.50%
[Val Step 4] Loss: 1.2649 | Top-1: 72.27% | Top-5: 87.89%
[Val Step 5] Loss: 1.2572 | Top-1: 72.19% | Top-5: 88.12%
[Val Step 6] Loss: 1.3002 | Top-1: 71.61% | Top-5: 87.50%
[Val Step 7] Loss: 1.3136 | Top-1: 72.10% | Top-5: 87.72%


KeyboardInterrupt: 

In [ ]:
torch.save(custom_model.state_dict(), "final_weights15.pth")

drive_path = "/content/drive/MyDrive/final_weights15.pth"
shutil.copy("final_weights15.pth", drive_path)

'/content/drive/MyDrive/final_weights15.pth'

In [ ]:
def process_scale_keys(input_path, output_path):
    # 1. تحميل ملف الـ checkpoint
    checkpoint = torch.load(input_path, map_location='cpu')

    # 2. التحقق مما إذا كان الملف قاموساً مباشراً أو يحتوي على 'state_dict'
    # غالباً ملفات الـ pth تكون عبارة عن state_dict
    state_dict = checkpoint.get('state_dict', checkpoint)

    processed_count = 0

    # 3. تعديل القيم المطلوبة
    for key in state_dict.keys():
        if 'scale' in key.lower() and key.lower() != 'input_scale' and key.lower() != 'output_scale':
            # التأكد من أن القيمة عبارة عن Tensor
            value = state_dict[key]

            # حساب log2 ثم التحويل إلى int8
            # ملاحظة: نستخدم clamp أو rounding إذا لزم الأمر قبل التحويل
            log_scaled = torch.log2(value)
            state_dict[key] = log_scaled.to(torch.int8)

            processed_count += 1
            print(f"Updated key: {key}")

    # 4. حفظ الملف الجديد
    torch.save(checkpoint, output_path)
    print(f"\nDone! Processed {processed_count} keys.")
    print(f"Saved to: {output_path}")

# استخدام الكود
process_scale_keys('final_weights12.pth', 'final_weights13.pth')

Updated key: input_norm_output_scale
Updated key: patch_embed.fused_proj_output_scale
Updated key: layers.0.blocks.0.attn.q_output_scale
Updated key: layers.0.blocks.0.attn.k_output_scale
Updated key: layers.0.blocks.0.attn.v_output_scale
Updated key: layers.0.blocks.0.attn.qk_output_scale
Updated key: layers.0.blocks.0.attn.attn_v_output_scale
Updated key: layers.0.blocks.0.attn.proj_output_scale
Updated key: layers.0.blocks.0.mlp.fuse_linear_bn1_output_scale
Updated key: layers.0.blocks.0.mlp.fuse_linear_bn2_output_scale
Updated key: layers.0.blocks.1.attn.q_output_scale
Updated key: layers.0.blocks.1.attn.k_output_scale
Updated key: layers.0.blocks.1.attn.v_output_scale
Updated key: layers.0.blocks.1.attn.qk_output_scale
Updated key: layers.0.blocks.1.attn.attn_v_output_scale
Updated key: layers.0.blocks.1.attn.proj_output_scale
Updated key: layers.0.blocks.1.mlp.fuse_linear_bn1_output_scale
Updated key: layers.0.blocks.1.mlp.fuse_linear_bn2_output_scale
Updated key: layers.0.downsa

In [26]:
orig_state = torch.load("/content/drive/MyDrive/final_weights15.pth")
with open("final_weights15.txt", "w") as f:
    for name, param in orig_state.items():
        f.write(f"size: {param.shape}\n")
        f.write(f"{name}: {param}\n\n")

drive_path = "/content/drive/MyDrive/final_weights15.txt"
shutil.copy("final_weights15.txt", drive_path)

'/content/drive/MyDrive/final_weights15.txt'